In [ ]:
import warnings
from config import *
import os
import re
import pandas as pd
import numpy as np

# Show all rows
pd.set_option('display.max_rows', None)
# Show all columns
pd.set_option('display.max_columns', None)
# Do not truncate column values
pd.set_option('display.max_colwidth', None)

In [ ]:
randhrs = "data/original data/randhrs1992_2020v2.dta"
hrs_tracker = "data/original data/trk2022tr_r.dta"

In [ ]:
# Set wave e.g:
# year = 2 * (wave - 5) from 2002 to 2016

# Run once per wave; set the HRS_WAVE environment variable to choose it (e.g. HRS_WAVE=10 for 2010).
wave = int(os.environ.get('HRS_WAVE', 13))
year = 2 * (wave - 5)

In [ ]:
# Time-invariant variables (from base interview)
timeinvariant_household_col = ['hidpn']
timeinvariant_respondent_col = ['gender', 'edyrs', 'bplace']
timeinvariant_col = [f'h{col}' for col in timeinvariant_household_col] + \
                    [f'ra{col}' for col in timeinvariant_respondent_col]

# Wave-specific variables for respondent and household
wave_household_col = ['child'] # Number of children

wave_respondent_col = [
    'lbrf',     # Labor force status
    'shlt',     # Self-rated health
    'agey_m',   # Age in completed years at interview (mid-month of interview); RwAGEY_M
    'height',
    'weight',
    'smokev',   # Smoker status
    'proxy',    # Proxy interview indicator
    'imrc',  # immediate recall of a list of 10 words
    'dlrc',  # delayed recall of a list of 10 words
    'ser7',  # five trials of serial 7s
    'bwc20',  # backward counting
    'prmem',  # proxy rating of respondent memory
    'iadl5a',  # iadl5a = sum(phonea, moneya, medsa, shopa, mealsa), referred as "iadlza" in appendix
    'cendiv', # Census Division
    'mstat', # Marital Status
    'effort', # CESD Everything an effort
    'hibpe', # Ever had high blood pressure
    'diabe', # Ever had diabetes
    'cancre', # Ever had cancer
    'lunge', # Ever had lung disease
    'hearte', # Ever had heart problems
    'stroke', # Ever had stroke
    'arthre', # Ever had arthritis
    'slfmem', # Self-rated memory
    'livpar', # Number of living parents
    'momage', # Mother age current/at death
    'dadage', # Father age current/at death
    'livsib', # Number of living siblings
    'hlthlm', # Health problems limit work
    'hosp',     # Any overnight hospital stay, previous 2 years (yes/no); RwHOSP
    'nrshom', # Any overnight nursing home stay, previous 2 years (yes/no); RwNRSHOM. Not current residence (RwNHMLIV)
    'nrstim', # Number of nursing home stays, previous 2 years; RwNRSTIM
    'nrsnit', # Number of nights in a nursing home over all stays, previous 2 years; RwNRSNIT. Not hospital nights
    'doctim', # Doctor visits (reported number of visits)
    'depres', # CESD Felt depressed
    'sleepr', # CESD Sleep was restless
    'whappy', # CESD Was happy
    'flone',    # CESD Felt lonely
    'fsad',     # CESD Felt sad
    'going',    # CESD Could not get going
    'enlife',   # CESD Enjoyed life
    'drink',    # Ever drinks any alcohol
    'smoken',   # Smokes now
    'toilta',   # Some Difficulty-Using the toilet
    'adl5a',    # Number of ADLs with any difficulty /0-5: bathing, dressing, eating, getting in/out of bed, walking across a room; RwADL5A
    'mapa',     # Any difficulty using a map (an IADL); RwMAPA. NOT use of a mobility aid
    'walksa',   # Some Difficulty-Walk sev blocks
    'walk1a',   # Some Difficulty-Walk one block
    'sita',     # Some Difficulty-Sit for 2 hours
    'chaira',   # Some Difficulty-Get up fr chair
    'climsa',   # Some Difficulty-Clmb sev flt str
    'clim1a',   # Some Difficulty-Clmb 1 flt stair
    'stoopa',   # Some Difficulty-Stoop/Kneel/Crch
    'lifta',    # Some Difficulty-Lift/carry 10 lbs
    'dimea',    # Some Difficulty-Pick up a dime
    'armsa',    # Some Difficulty-Rch/xtnd arms up
    'pusha',    # Some Difficulty-Push/pull lg obj
    'mobila',   # Some Difficulty-Mobility /0-5
    'lgmusa',   # Some Diff-Large Muscle /0-4
    'grossa',   # Walk1/R,Clim1,Bed,Bath/0-5
    'finea',    # Dime/Eat/Dress /0-3
]

if wave==6:
    wave_respondent_col.append('vigact') # (For w6): Whether vigorous phys act 3+/wk
else:
    wave_respondent_col.append('vgactx') # (For w7-w15) Whether vigorous phys act 3+/wk, Freq vigorous phys active {finer scale}

# Construct final columns to load
columns = timeinvariant_col + \
          [f'h{wave}{col}' for col in wave_household_col] + \
          [f'r{wave}{col}' for col in wave_respondent_col]

# Read the selected columns from the dataset
chunks_main = pd.read_stata(randhrs, columns=columns)

# Map the correct 'prfin' column based on year
col_prfin = {2:'HA011',
             4:'JA011',
             6:'KA011',
             8:'LA011',
             10:'MA011',
             12: 'NA011',
             14: 'OA011',
             16: 'PA011',
             18: 'QA011',
             20: 'RA011',
             }
original_prfin_col = col_prfin[year]
new_prfin_col = f'r{wave}prfin'

# Load prefin column dataset 
prfin_data = pd.read_csv('data/original data/'+f'h{year}A_R.csv')

In [ ]:
# Create unique person ID
#prfin_data['hhidpn'] = (prfin_data['HHID'] + prfin_data['PN']).astype(int)
prfin_data['hhidpn']=(prfin_data['HHID'].astype(str)+'0'+prfin_data['PN'].astype(str)).astype(int)

# Recode original prfin values into categories
prfin_data[new_prfin_col] = pd.cut(
    prfin_data[original_prfin_col],
    bins=[0, 1, 2, 3],
    labels=['1.none', '2.some', '3.prevented'],
    ordered=True
)

# Keep only hhidpn and recoded prfin variable
prfin_data = prfin_data[['hhidpn', new_prfin_col]]

# Merge with main dataset on hhidpn
chunks = pd.merge(
    chunks_main,
    prfin_data,
    on='hhidpn',
    how='left'
)

In [ ]:
# Define new column name for race/ethnicity based on wave
raceeth_col = f'r{wave}raceeth'

# Load relevant columns from HRS tracker file
raceeth_tracker = pd.read_stata(
    hrs_tracker,
    columns=['HHID', 'PN', 'RACE', 'HISPANIC', 'NIWWAVE']
)

# Generate unique person ID
raceeth_tracker['hhidpn'] = (raceeth_tracker['HHID'] + raceeth_tracker['PN']).astype(int)

# Recode race/ethnicity following HRS coding logic:
# 0 = Non-Hispanic White
# 1 = Non-Hispanic Black
# 2 = Hispanic
# 3 = Non-Hispanic Other
# Tracker codes: HISPANIC 1-3 = Hispanic, 5 = not Hispanic, 0 = unknown; RACE 1 = White, 2 = Black, 7 = other, 0 = unknown.
# Unknown Hispanic status or unknown race stays missing (NaN).
raceeth_tracker.loc[raceeth_tracker['HISPANIC'].isin([1, 2, 3]), raceeth_col] = 2
raceeth_tracker.loc[(raceeth_tracker['HISPANIC'] == 5) & (raceeth_tracker['RACE'] == 1), raceeth_col] = 0
raceeth_tracker.loc[(raceeth_tracker['HISPANIC'] == 5) & (raceeth_tracker['RACE'] == 2), raceeth_col] = 1
raceeth_tracker.loc[(raceeth_tracker['HISPANIC'] == 5) & (raceeth_tracker['RACE'] == 7), raceeth_col] = 3

# Race/ethnicity is time-invariant and the tracker has one row per person, so no wave filter is applied.

# Drop original identifiers and raw race/ethnicity variables
raceeth_tracker.drop(columns=[ 'HISPANIC', 'RACE'], inplace=True)

# Check distribution of the recoded race/ethnicity variable
raceeth_tracker[raceeth_col].value_counts(dropna=False)


In [ ]:
chunks.describe(include='all')

In [ ]:
# Merge race/ethnicity tracker data into the main dataset using `hhidpn` as the key
chunks = pd.merge(
    chunks,                # main analysis dataset (already includes wave, prfin, etc.)
    raceeth_tracker,       # race/ethnicity tracker data
    on='hhidpn',           # merge key
    how='left'             # keep all cases from main dataset
)
chunks.head()

In [ ]:
# Convert time-invariant categorical variables to float (delete float setting)
timeinvariant_categorical_cols = ['ragender', 'raedyrs']

for col in timeinvariant_categorical_cols:
    chunks[col] = chunks[col].cat.codes.astype(float)
    chunks.loc[chunks[col] == -1, col] = float('nan')

# Recode one-hot indicator
for col in [
    'smokev', 'proxy', 'prmem', 'prfin','bwc20',  'effort',
    'hibpe', 'diabe', 'cancre', 'lunge', 'hearte', 'stroke',
    'arthre', 'slfmem', 'hlthlm', 'hosp', 'nrshom',
    'depres','sleepr', 'whappy', 'flone', 'fsad', 'going',
    'enlife', 'drink', 'smoken', 'toilta', 'mapa', 'walksa',
    'walk1a', 'sita', 'chaira', 'climsa', 'clim1a', 'stoopa',
    'lifta', 'dimea', 'armsa','pusha'
]:

# delete categorical variables: 'cendiv', 'mstat', 'bplace' due to non-ordinal

    wave_col = f'r{wave}{col}'
    chunks.loc[:, wave_col] = chunks[wave_col].cat.codes.astype(float)
    chunks.loc[chunks[wave_col] == -1, wave_col] = float('nan')

# Harmonize vigorous physical activity to the 2000 baseline definition
# (RAND R5VIGACT: vigorous activity 3+ times/week, 0 = no, 1 = yes).
# Wave 6 has the same RwVIGACT. From wave 7 RAND only provides RwVGACTX, a frequency scale:
# 1.every day, 2.>1 per week, 3.1 per week, 4.1-3 per mon, 5.never.
# "Every day" or ">1 per week" -> 1, otherwise 0. ">1 per week" is the closest category to
# "3+/week" and is slightly broader.
if wave == 6:
    vig = chunks[f'r{wave}vigact'].cat.codes.astype(float)
else:
    vig = chunks[f'r{wave}vgactx'].cat.codes.astype(float)  # 0 = every day ... 4 = never, -1 = missing
    vig = vig.where(vig == -1, (vig <= 1).astype(float))
    chunks.drop(columns=[f'r{wave}vgactx'], inplace=True)
chunks[f'r{wave}vigact'] = vig.where(vig != -1, float('nan'))

# Bin education into stage groups
raedstg: ['1.0-7', '2.8-11', '3.12', '4.13+']
chunks['raedstg'] = pd.cut(
    chunks['raedyrs'],
    bins=[-1, 7, 11, 12, chunks['raedyrs'].max()],
    labels=['1.0-7', '2.8-11', '3.12', '4.13+']
)

# Convert all wave-specific categorical variables to float
# lbrf (labor force status) is nominal, so it stays categorical and is one-hot encoded with the
# other nominal variables before saving.
wave_categorical_cols = [
    'shlt',     # Self-rated health
]

for col in wave_categorical_cols:
    wave_col = f'r{wave}{col}'
    chunks[wave_col] = chunks[wave_col].cat.codes.astype(float)
    chunks.loc[chunks[wave_col] == -1, wave_col] = float('nan')

In [ ]:
# Drop the original 'raedyrs' column (already binned into 'raedstg' and encoded into 'redleq17')
chunks.drop(columns=['raedyrs'], inplace=True)

# (Optional) Reorder columns: Move key variables like hhidpn and race/ethnicity to the front
cols_order = ['hhidpn']
if f'r{wave}raceeth' in chunks.columns:
    cols_order.append(f'r{wave}raceeth')
if 'ragender' in chunks.columns:
    cols_order.append('ragender')
if 'raedstg' in chunks.columns:
    cols_order.append('raedstg')
if 'redleq17' in chunks.columns:
    cols_order.append('redleq17')

# Add remaining columns after key ones
cols_remaining = [col for col in chunks.columns if col not in cols_order]
chunks = chunks[cols_order + cols_remaining]
chunks[f'r{wave}raceeth'].value_counts(dropna=False)

In [ ]:
# Define variable names
proxy_col = f'r{wave}proxy'
demscr_col = f'r{wave}demscr'
demcls_col = f'r{wave}demcls'

# Define column groups used to calculate cognitive score
self_dem_cols_wave = [f'r{wave}{col}' for col in SELF_DEM_COLS]
proxy_dem_cols_wave = [f'r{wave}{col}' for col in PROXY_DEM_COLS]

# Recode for self-respondents
self_mask = (chunks[proxy_col] == 0)
print(self_mask.sum())
chunks.loc[self_mask, demscr_col] = chunks.loc[self_mask, self_dem_cols_wave].sum(
    axis=1, min_count=len(self_dem_cols_wave)
)
chunks.loc[self_mask & (chunks[demscr_col] <= 6), demcls_col] = 1.0
chunks.loc[self_mask & (chunks[demscr_col] > 6), demcls_col] = 0.0

# Recode for proxy-respondents
proxy_mask = (chunks[proxy_col] == 1)
chunks.loc[proxy_mask, demscr_col] = chunks.loc[proxy_mask, proxy_dem_cols_wave].sum(
    axis=1, min_count=len(proxy_dem_cols_wave)
)
chunks.loc[proxy_mask & (chunks[demscr_col] >= 6), demcls_col] = 1.0
chunks.loc[proxy_mask & (chunks[demscr_col] < 6), demcls_col] = 0.0

# Drop rows with missing demscr (as they can't be classified)
# chunks.dropna(subset=[demscr_col], inplace=True)

# Drop raw demscr score now that demcls has been derived
chunks.drop(columns=[demscr_col], inplace=True)

In [ ]:
def remove_wave_prefix(col, wave):
    # Remove 'ra', 'r11', 'ha', 'h11' prefixes only
    return re.sub(rf'^(r|h)(a|{wave})', '', col)

# Apply clean renaming to all columns
chunks.rename(
    columns={col: remove_wave_prefix(col, wave) for col in chunks.columns},
    inplace=True
)

In [ ]:
core_feature_cols = [
    'HHID', 'PN', 'hhidpn', 'NIWWAVE', # keys to merge, not x variables
    # all seleted x variables
    'edstg', "child", "lbrf", "shlt", "agey_m", "height", "weight", "smokev", "proxy", "cendiv", "mstat", "effort", "hibpe", "diabe",
    'vigact',
    "slfmem", "livpar", "momage", "dadage", "livsib", "hlthlm", "hosp", "nrshom", "nrstim", "nrsnit", "doctim", "depres", "sleepr", "whappy", "flone", "fsad", "going", "enlife", "drink", "smoken", "cancre", "lunge", "hearte", "stroke", "arthre", "toilta",
    "adl5a", "mapa", "walksa", "walk1a", "sita", "chaira", "climsa", "clim1a", "stoopa", "lifta", "dimea", "armsa", "pusha",
    "mobila", "lgmusa", "grossa", "finea", "raceeth","gender",
    "demcls" # y: Binary cognitive impairment classification
]

chunks_new=chunks[core_feature_cols]
# optional: dropping all rows with age ≤ 50, and demcls == 1, exclude all people who has dementia and age less than or equal to 50 on baseline year (2000,2006,2008,2010) to predict
#chunks_new = chunks_new[chunks_new['agey_m'] > 50].copy().reset_index(drop=True)
#chunks_new = chunks_new[chunks_new['demcls'] != 1].copy().reset_index(drop=True)

# Convert the 'raceeth' column to a categorical variable
chunks_new['raceeth'] = chunks_new['raceeth'].astype('category')

In [ ]:
if year<10:
    year=f'0{year}'

In [ ]:
# Functional items: keep the HRS core answer "7. don't do" as its own category.
# RAND sets "don't do" to missing, but those people have a higher dementia risk than people reporting difficulty.
# Where RAND is missing and the core answer is "don't do": <item> = 0 and <item>_dontdo = 1.
# <item>_dontdo = 0 when the item was answered, and stays missing when the item is missing for another reason.
# toilta is not included: it has only a handful of "don't do" answers and RAND gives some of them an observed value.
from dont_do import read_core_answers, CORE_ITEMS, DONT_DO
if wave in CORE_ITEMS:
    core = read_core_answers(wave)
    dd_items = [i for i in CORE_ITEMS[wave][2] if i != 'toilta']
    core = chunks_new[['hhidpn']].merge(core[['hhidpn'] + dd_items], on='hhidpn', how='left')
    assert (core['hhidpn'].to_numpy() == chunks_new['hhidpn'].to_numpy()).all()
    for item in dd_items:
        is_dd = (chunks_new[item].isna() & (core[item] == DONT_DO).to_numpy()).to_numpy()
        chunks_new[f'{item}_dontdo'] = np.where(is_dd, 1.0, np.where(chunks_new[item].notna(), 0.0, np.nan))
        chunks_new.loc[is_dd, item] = 0.0
        print(f"{item}: {is_dd.sum()} don't-do answers recoded")
else:
    print(f"Wave {wave}: no core-file mapping for don't-do answers, so no <item>_dontdo columns are created.")

# RAND's combined code 6.separated/divorced is used when separated vs divorced is unknown. It is too rare to be
# its own category (1 person in the 2000 cohort), so it is set to missing and imputed.
if '6.separated/divorced' in chunks_new['mstat'].cat.categories:  # not a category in every wave
    chunks_new['mstat'] = chunks_new['mstat'].cat.remove_categories(['6.separated/divorced'])

# One-hot encode nominal variables. A missing category stays missing in all of its dummy columns;
# pd.get_dummies alone gives all zeros, i.e. silently assigns the reference category.
cat_cols = chunks_new.select_dtypes(include=['category']).columns.tolist()
cat_missing = chunks_new[cat_cols].isna()
chunks_new = pd.get_dummies(chunks_new, columns=cat_cols, drop_first=True, dtype=float)
for col in cat_cols:
    dummy_cols = [c for c in chunks_new.columns if c.startswith(f'{col}_')]
    chunks_new.loc[cat_missing[col], dummy_cols] = float('nan')
chunks_new.to_csv('data/preprocessed data/'+f'20{year}.csv',na_rep="NA" ,index=False)

In [ ]:
#Select 'demcls' and rename it to '__demcls' and HHID,PN, hhidpn
chunks_new.rename(columns={'demcls':f'{year}demcls'}, inplace=True)
y = chunks_new[['HHID','PN', 'hhidpn' , f'{year}demcls']]
y.to_csv('data/preprocessed data/'+f'_{year}y.csv',na_rep="NA" ,index=False)